# 03 - M3: backbone + LoRA (gate on the real CLIP ViT-L/14@336)
**Runtime > T4 GPU.** Gate: taps are 2 x (B,576,1024) with CLS dropped; LoRA tensors get gradients, base CLIP gets none. Also checks: at init the wrapped model equals frozen CLIP exactly, trainable count = 786,432, and memory/speed at batch 8 (fp16 autocast).

In [ ]:
# BOOTSTRAP - identical first cell in every notebook. Safe to re-run: clones the repo if missing, otherwise pulls.
import os, subprocess
from google.colab import drive, userdata
drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/csm-clip'
os.makedirs(DRIVE_ROOT + '/runs', exist_ok=True)
GH_USER = 'bhargav-reddy-nasappagari'
try:
    tok = userdata.get('GITHUB_TOKEN')      # only needed if the repo is private
except Exception:
    tok = None
url = f"https://{tok + '@' if tok else ''}github.com/{GH_USER}/csm-clip.git"
if not os.path.exists('/content/csm-clip'):
    subprocess.run(['git', 'clone', '-q', url, '/content/csm-clip'], check=True)
else:
    subprocess.run(['git', '-C', '/content/csm-clip', 'pull', '-q', '--rebase', url, 'master'], check=True)
os.chdir('/content/csm-clip')
subprocess.run(['git', 'remote', 'set-url', 'origin', f'https://github.com/{GH_USER}/csm-clip.git'], check=True)  # keep token out of .git/config
subprocess.run(['pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
print(subprocess.run(['git', 'log', '--oneline', '-3'], capture_output=True, text=True).stdout)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python -m pytest -q

In [ ]:
import torch, numpy as np
from datasets import load_dataset
from transformers import CLIPVisionModel
from src.data import load_manifest, ManifestDataset
from src.backbone import ClipLoRABackbone

m = load_manifest('Midjourney')
ds = load_dataset('TheKernel01/Tiny-GenImage', revision=m['revision'])
clip = CLIPVisionModel.from_pretrained('openai/clip-vit-large-patch14-336', torch_dtype=torch.float32).cuda().eval()
# reference: untouched frozen CLIP features for the same batch (before LoRA is injected)
tr = ManifestDataset(m, 'train', ds['train'], ds['validation'], protocol='controlled', train=False)
x = torch.stack([tr[i][0] for i in range(8)]).cuda()
with torch.no_grad():
    ref = clip(pixel_values=x, output_hidden_states=True).hidden_states
    ref16, ref24 = ref[16][:, 1:].clone(), ref[24][:, 1:].clone()
bb = ClipLoRABackbone(clip, taps=(16, 24), r=8, alpha=16.0).cuda()
print(bb.param_report())

In [ ]:
# GATE 1: shapes, CLS dropped, init == frozen CLIP, trainable count
with torch.no_grad():
    a, b = bb(x)
assert a.shape == b.shape == (8, 576, 1024), (a.shape, b.shape)
d16, d24 = (a - ref16).abs().max().item(), (b - ref24).abs().max().item()
print('max |diff| vs frozen CLIP at init:', d16, d24)
assert d16 < 1e-4 and d24 < 1e-4
rep = bb.param_report()
assert rep['trainable'] == 786_432 and rep['lora_modules'] == 48, rep
print('GATE 1 OK')

In [ ]:
# GATE 2: gradients. Step 1: B gets grad (A's is exactly 0 because B=0). Step 2: after B moves, everything gets grad.
bb.zero_grad(set_to_none=True)
with torch.autocast('cuda', dtype=torch.float16):
    a, b = bb(x)
(a.float().mean() + b.float().pow(2).mean()).backward()
bad = [n for n, p in bb.named_parameters() if 'lora_' not in n and p.grad is not None]
assert not bad, bad[:3]
assert all(p.grad is not None and p.grad.abs().sum() > 0 for n, p in bb.named_parameters() if 'lora_B' in n)
with torch.no_grad():
    for n, p in bb.named_parameters():
        if 'lora_B' in n: p.add_(0.01 * torch.randn_like(p))
bb.zero_grad(set_to_none=True)
with torch.autocast('cuda', dtype=torch.float16):
    a, b = bb(x)
(a.float().mean() + b.float().pow(2).mean()).backward()
nz = [(n, p.grad.abs().sum().item()) for n, p in bb.named_parameters() if 'lora_' in n]
assert all(g > 0 for _, g in nz) and not any(p.grad is not None for n, p in bb.named_parameters() if 'lora_' not in n)
print('lora tensors with grad:', len(nz), '| base tensors with grad: 0')
print('GATE 2 OK')

In [ ]:
# Budget probe (not a gate): peak memory and speed, forward+backward, batch 8, fp16 autocast
import time
bb.zero_grad(set_to_none=True); torch.cuda.reset_peak_memory_stats(); torch.cuda.synchronize()
t = time.time()
for _ in range(3):
    with torch.autocast('cuda', dtype=torch.float16):
        a, b = bb(x)
    (a.float().mean() + b.float().mean()).backward()
torch.cuda.synchronize()
print(f'peak GPU mem {torch.cuda.max_memory_allocated()/2**30:.2f} GiB | {3*8/(time.time()-t):.1f} img/s (fwd+bwd, backbone only)')

In [ ]:
# Save the gate result next to the other runs
import json
out = {'milestone': 'M3', 'taps': [16, 24], 'shape': [8, 576, 1024], 'init_max_diff': [d16, d24],
       'param_report': rep, 'lora': {'r': 8, 'alpha': 16.0, 'targets': ['q_proj', 'v_proj']}, 'gate': 'PASS'}
for p in (f'{DRIVE_ROOT}/runs/m3_gate.json', '/content/csm-clip/runs/m3_gate.json'):
    json.dump(out, open(p, 'w'), indent=1)
print(out)